In [1]:
import json
import time
import ollama
import chromadb
from chromadb.config import Settings
from ollama._types import EmbeddingsResponse
from collections import Counter
from typing import List

# OneOrMany = Union[T, List[T]]
# --- Config ---
CHUNKS_PATH = "./chunks_all.json"
CHROMA_DB_PATH = "./chroma_db"  # persisted locally
COLLECTION_NAME = "socratic_tutor_collection"
EMBED_MODEL = "nomic-embed-text"  # ollama pull nomic-embed-text
BATCH_SIZE = 50  # chunks per embedding batch

---
## 2. Load & Clean Chunks

In [2]:
with open(CHUNKS_PATH, encoding="utf-8") as f:
    raw_chunks = json.load(f)

print(f"Raw chunks loaded: {len(raw_chunks)}")

# Filter chunks too short to be meaningful (noise)
MIN_LENGTH = 100
chunks = [c for c in raw_chunks if len(c["content"]) >= MIN_LENGTH]

print(f"After filtering  : {len(chunks)}")
print(f"Removed          : {len(raw_chunks) - len(chunks)}")


dist = Counter(c["id"] for c in chunks)
for ch, count in sorted(dist.items()):
    has_code = sum(1 for c in chunks if c["id"] == ch and c["has_code"])
    print(f"  {ch:<20} total={count:<5} with_code={has_code}")

Raw chunks loaded: 2725
After filtering  : 2637
Removed          : 88
  chunk_00000          total=1     with_code=0
  chunk_00001          total=1     with_code=1
  chunk_00002          total=1     with_code=0
  chunk_00003          total=1     with_code=1
  chunk_00004          total=1     with_code=0
  chunk_00005          total=1     with_code=1
  chunk_00006          total=1     with_code=0
  chunk_00007          total=1     with_code=1
  chunk_00009          total=1     with_code=1
  chunk_00010          total=1     with_code=0
  chunk_00011          total=1     with_code=1
  chunk_00012          total=1     with_code=0
  chunk_00013          total=1     with_code=1
  chunk_00015          total=1     with_code=1
  chunk_00016          total=1     with_code=0
  chunk_00017          total=1     with_code=1
  chunk_00018          total=1     with_code=0
  chunk_00019          total=1     with_code=1
  chunk_00020          total=1     with_code=1
  chunk_00022          total=1     wi

## 3. Embed & Ingest into ChromaDB


In [3]:
def get_embedding(text: str, is_query: bool = False) -> EmbeddingsResponse:
    """Get embedding with correct nomic-embed-text prompt prefixes."""
    prefix = "search_query: " if is_query else "search_document: "
    response = ollama.embeddings(model=EMBED_MODEL, prompt=prefix + text)
    return response["embedding"]


def embed_batch(texts: List[str], is_query: bool = False) -> List[EmbeddingsResponse]:
    return [get_embedding(t, is_query=is_query) for t in texts]


In [4]:
# Initialize ChromaDB (persistent local)
client = chromadb.PersistentClient(
    path=CHROMA_DB_PATH, settings=Settings(anonymized_telemetry=False)
)

# Drop and recreate collection for clean ingestion
# Comment out the delete line if you want to resume a partial ingestion
try:
    client.delete_collection(COLLECTION_NAME)
    print(f"Deleted existing collection '{COLLECTION_NAME}'")
except:
    pass

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},  # cosine distance for semantic search
)
print(f"Collection '{COLLECTION_NAME}' created.")

Collection 'socratic_tutor_collection' created.


In [5]:
import time as _time

# ── Pre-flight token guard ───────────────────────────────────────────────────────────────────────
# Use the same tokenizer as the chunking notebook to pre-screen chunks.
# EMBED_MAX_TOKENS = 8138 (8192 hard limit - 4 prefix tokens - 50 safety margin)
_EMBED_PREFIX = "search_document: "
_EMBED_MAX = 8138

try:
    import tiktoken as _tiktoken
    _tok = _tiktoken.get_encoding("cl100k_base")
    def _embed_token_count(text: str) -> int:
        return len(_tok.encode(_EMBED_PREFIX + text))
except ImportError:
    # Fallback: conservative character-based estimate (4 chars ≈ 1 token)
    def _embed_token_count(text: str) -> int:
        return len(_EMBED_PREFIX + text) // 4

pre_skipped = []
safe_chunks = []
for c in chunks:
    et = _embed_token_count(c["content"])
    if et > _EMBED_MAX:
        pre_skipped.append({"id": c["id"], "embed_tokens": et, "chars": len(c["content"])})
    else:
        safe_chunks.append(c)

if pre_skipped:
    print(f"Pre-flight: skipping {len(pre_skipped)} chunks that exceed embed limit ({_EMBED_MAX}t):")
    for s in pre_skipped:
        print(f"  {s['id']} | embed={s['embed_tokens']}t | chars={s['chars']}")
else:
    print(f"Pre-flight: all {len(chunks)} chunks within embed limit \u2714")

chunks_to_ingest = safe_chunks
print(f"Chunks to ingest: {len(chunks_to_ingest)}")

Pre-flight: all 2637 chunks within embed limit ✔
Chunks to ingest: 2637


In [6]:
import time as _time

MAX_RETRIES = 2
RETRY_DELAY = 2.0  # seconds, doubled on each retry

def embed_with_retry(text: str, is_query: bool = False) -> list:
    """Embed a single text with exponential backoff retry for transient errors."""
    last_err = None
    for attempt in range(MAX_RETRIES + 1):
        try:
            return get_embedding(text, is_query=is_query)
        except Exception as e:
            err_str = str(e).lower()
            # Don't retry oversized content errors — they will never succeed
            if "input length exceeds" in err_str or "context length" in err_str:
                raise
            last_err = e
            if attempt < MAX_RETRIES:
                wait = RETRY_DELAY * (2 ** attempt)
                print(f"  Transient error (attempt {attempt+1}/{MAX_RETRIES+1}), retrying in {wait:.1f}s: {e}")
                _time.sleep(wait)
    raise last_err

total = len(chunks_to_ingest)
ingested = 0
skipped = []
errors = []

start = _time.time()

for idx, c in enumerate(chunks_to_ingest):
    try:
        embedding = embed_with_retry(c["content"], is_query=False)
        metadata = {
            "id": c["id"],
            "heading": c.get("heading", ""),
            "has_code": str(c.get("has_code", False)),
        }
        collection.add(
            ids=[c["id"]],
            embeddings=[embedding],
            documents=[c["content"]],
            metadatas=[metadata],
        )
        ingested += 1
        if ingested % BATCH_SIZE == 0 or ingested == total:
            elapsed = _time.time() - start
            rate = ingested / elapsed if elapsed > 0 else 0
            eta = (total - ingested) / rate if rate > 0 else 0
            print(f"[{ingested:>4}/{total}] {rate:.1f} chunks/s | ETA: {eta:.0f}s")

    except Exception as ex:
        err_str = str(ex)
        skipped.append({"id": c["id"], "error": err_str, "chars": len(c["content"])})
        errors.append({"id": c["id"], "error": err_str})
        print(f"  SKIP {c['id']}: {err_str[:120]}")

print(f"\nIngestion complete.")
print(f"  Ingested : {ingested}/{total}")
print(f"  Skipped  : {len(skipped)}")
if skipped:
    for s in skipped:
        print(f"    {s['id']} | chars={s['chars']} | {s['error'][:80]}")
print(f"  Total time: {_time.time() - start:.1f}s")
print(f"  Collection count: {collection.count()}")

[  50/2637] 18.2 chunks/s | ETA: 142s
[ 100/2637] 19.2 chunks/s | ETA: 132s
[ 150/2637] 17.2 chunks/s | ETA: 145s
  SKIP chunk_00213: the input length exceeds the context length (status code: 500)
[ 200/2637] 16.7 chunks/s | ETA: 146s
[ 250/2637] 16.7 chunks/s | ETA: 143s
[ 300/2637] 16.6 chunks/s | ETA: 141s
[ 350/2637] 16.3 chunks/s | ETA: 140s
[ 400/2637] 16.3 chunks/s | ETA: 137s
[ 450/2637] 16.6 chunks/s | ETA: 132s
[ 500/2637] 16.5 chunks/s | ETA: 129s
[ 550/2637] 16.4 chunks/s | ETA: 127s
[ 600/2637] 16.1 chunks/s | ETA: 126s
[ 650/2637] 15.6 chunks/s | ETA: 127s
[ 700/2637] 15.5 chunks/s | ETA: 125s
[ 750/2637] 15.6 chunks/s | ETA: 121s
[ 800/2637] 15.6 chunks/s | ETA: 118s
[ 850/2637] 15.3 chunks/s | ETA: 117s
[ 900/2637] 15.3 chunks/s | ETA: 114s
  SKIP chunk_01061: the input length exceeds the context length (status code: 500)
  SKIP chunk_01070: the input length exceeds the context length (status code: 500)
  SKIP chunk_01074: the input length exceeds the context length (st

## 4. Retrieval Validation

In [7]:
def query_collection(query: str, n_results: int = 5, only_code: bool = False) -> None:
    """Query ChromaDB and print results with similarity scores."""
    where = {"has_code": "True"} if only_code else None

    query_emb = get_embedding(query, is_query=True)

    results = collection.query(
        query_embeddings=[query_emb],
        n_results=n_results,
        where=where,
        include=["documents", "metadatas", "distances"],
    )

    print(f"\nQuery: '{query}'")
    print(f"{'─' * 70}")

    for i, (doc, meta, dist) in enumerate(
        zip(results["documents"][0], results["metadatas"][0], results["distances"][0])
    ):
        similarity = 1 - dist  # ChromaDB cosine returns distance, not similarity
        has_code = meta["has_code"] == "True"
        print(
            f"[{i + 1}] sim={similarity:.4f} | {meta['id']} | heading: '{meta['heading']}'"
        )
        print(f"     has_code={has_code}")
        print(f"     {doc}...")
        print()

In [8]:
# --- Validation queries ---
# These cover different concept types: memory, syntax, data structures, control flow
validation_queries = [
    "How do pointers work in C?",
    "What is the difference between malloc and calloc?",  
    "How do arrays relate to pointers?",  
    "What is a struct in C?", 
    "How does a for loop work in C?", 
    "What are function pointers?",
    "How do you handle strings in C?", 
    "What is undefined behavior in C?", 
]

for q in validation_queries:
    query_collection(q, n_results=3)


Query: 'How do pointers work in C?'
──────────────────────────────────────────────────────────────────────
[1] sim=0.8065 | chunk_02319 | heading: 'Here are some key points about pointers in C:'
     has_code=True
     ## Here are some key points about pointers in C:

*   **Declaration:** Pointers are declared using the asterisk (\*) symbol before the variable name. For example: *   **Initialization:** Pointers can be initialized with the address of another variable using the address-of operator (&). For example:

```c
int \*ptr;  // Pointer to an integer
char \*ch;   // Pointer to a character
float \*fp;  // Pointer to a float
```...

[2] sim=0.7985 | chunk_02321 | heading: 'Here are some key points about pointers in C:'
     has_code=True
     ## Here are some key points about pointers in C:

*   **Initialization:** Pointers can be initialized with the address of another variable using the address-of operator (&). For example: *   **Dereferencing:** Dereferencing a pointer means acc

In [9]:
query_collection("C string manipulation char array null terminator", n_results=3)
query_collection("string functions strlen strcpy C", n_results=3)


Query: 'C string manipulation char array null terminator'
──────────────────────────────────────────────────────────────────────
[1] sim=0.7396 | chunk_02410 | heading: 'here's how you can first declare an array and then assign characters one by one:'
     has_code=True
     ## here's how you can first declare an array and then assign characters one by one:

```c
// Define a character array for brand name
char brandName\[11\]; // Allocate space for 10 characters plus null terminator

// Assign characters one by one
brandName\[0\] = 'L';
brandName\[1\] = 'i';
brandName\[2\] = 't';
brandName\[3\] = ' ';
brandName\[4\] = 'M';
brandName\[5\] = 'e';
brandName\[6\] = 'n';
brandName\[7\] = 't';
brandName\[8\] = 'o';
brandName\[9\] = 'r';
brandName\[10\] = '\0'; // Null terminator to mark end of string

// Print the brand name
printf("Welcome to %s!\n", brandName);

return 0;
```...

[2] sim=0.7214 | chunk_02782 | heading: 'Two dimensional character array'
     has_code=True
     ## Two dimen

In [10]:
# Code-specific retrieval — only return chunks that have code examples
query_collection("pointer arithmetic example", n_results=3, only_code=True)
query_collection("malloc free memory allocation example", n_results=3, only_code=True)


Query: 'pointer arithmetic example'
──────────────────────────────────────────────────────────────────────
[1] sim=0.7943 | chunk_01804 | heading: 'Example of Function Pointers'
     has_code=True
     ## Example of Function Pointers

Here's a example of a function pointer. We create an **add** function that takes two **int arguments** and returns an **int**. We store its address in **func\_ptr** and call it with **3** and **5** as arguments using **func\_ptr(3, 5)**.

```c
return a + b;
```...

[2] sim=0.7815 | chunk_00496 | heading: '11.1.4Pointer validity'
     has_code=True
     ## 11.1.4Pointer validity

However, this last example may crash at the increment operation: A program execution that computes a pointer value outside the bounds of an array object (or one element beyond) fails.

```c
double A[2] = { 0.0, 1.0, };
double* p = &A[0];
printf("element %g\n", *p); // Referencing object
p += 3;                     // Invalid pointer addition
                            // Program